<div>
<center><img src="../assets/Flux-logo.svg" width="400"/>
</div>

# Chapter 5: Hybrid Quantum-Classical Workflows with Flux

In the previous chapter you saw Flux run classical HPC and AI/ML workloads and orchestrate user-space Kubernetes. In this chapter we add a *quantum* backend to the mix. We will run a **Quantum Approximate Optimization Algorithm (QAOA)** for graph max-cut, a hybrid workload where a classical optimizer running under Flux repeatedly evaluates a quantum circuit on a **quantum simulator**.

Everything in this chapter runs on a **local simulator** that ships with the [AWS Braket SDK](https://github.com/amazon-braket/amazon-braket-sdk-python). It executes right here on the VM, so there is:
- **no AWS account or credentials required**,
- **no network calls**, and
- **no cost**.

In this tutorial we will:
1. Confirm the local simulator is available
2. Run QAOA under Flux with `flux run`
3. Submit it non-blocking and inspect it with `flux submit` / `flux jobs`
4. Scale out to an **ensemble** of quantum simulations with Flux
5. (Advanced) Run the same job as a **container in Usernetes**
<br>

The classical part (building circuits, the COBYLA optimizer, computing the cut) and the quantum part (executing the circuit, sampling measurements) both run on your Flux-managed VM. This is exactly the hybrid quantum-classical pattern used in real research, and it maps naturally onto an HPC workload manager: Flux schedules the classical driver, and each circuit evaluation is a call into the quantum simulator.

The application here is a condensed, single-file version of the pipeline in [converged-computing/quantum-braket](https://github.com/converged-computing/quantum-braket).

If you haven't yet, open a Jupyter terminal alongside this notebook so you can copy-paste commands into it.

## Background: what is QAOA max-cut?

**Max-cut** asks: given a graph, partition its nodes into two groups so that the number of edges crossing between the groups (the "cut") is as large as possible. It is a classic NP-hard combinatorial optimization problem.

**QAOA** encodes each node as a qubit and builds a parameterized quantum circuit with two alternating layers: a *cost* layer (parameter `gamma`) and a *mixer* layer (parameter `beta`). Measuring the circuit yields bitstrings that assign each node to one of the two groups. A classical optimizer (here, COBYLA) adjusts `gamma` and `beta` to push the expected cut value higher.

Each optimizer iteration builds one circuit, runs it on the quantum backend, gets back measurement counts, and computes the average cut. That is the hybrid loop, and it is what Flux will schedule for us.

<div class="alert alert-block" style="background-color:#d4edda; color:#14532d">
<span style="font-weight:600">✅ Local and free.</span> We use the Braket SDK's built-in local state-vector simulator. It runs the identical circuits an AWS device would, but entirely on this VM. No credentials, no network, no cost. (If you later want to run on real AWS Braket hardware or the cloud SV1 simulator, the same script supports it with a flag — see the optional appendix at the end.)
</div>

## 1. Setup

Move into the chapter directory:

```bash
cd ~/tutorial/ch5
```

The Braket SDK and SciPy are pre-installed on the tutorial image. If for some reason they are missing, install them into your user environment (no AWS packages or configuration are needed):

```bash
pip install --user amazon-braket-sdk scipy
```

Confirm the local simulator works. This creates a tiny Bell-state circuit and samples it — a quick sanity check that runs in a fraction of a second, entirely locally:

```bash
python3 -c "
from braket.circuits import Circuit
from braket.devices import LocalSimulator
circ = Circuit().h(0).cnot(0, 1)
result = LocalSimulator().run(circ, shots=100).result()
print('Bell-state counts:', dict(result.measurement_counts))
"
```

You should see counts split between `00` and `11`, e.g. `{'00': 52, '11': 48}`. No AWS calls happen here.

## 2. Run QAOA under Flux

<div class="alert alert-block" style="background-color:skyblue; color:#0b2545">
<span style="font-weight:600">Description:</span> Run the full hybrid loop as a Flux job. The classical optimizer and the quantum simulation both run locally; Flux schedules the driver just like any other job.
</div>

We are already inside a Flux instance (the notebook was launched with `flux start`), so we can hand the driver to Flux. `flux run` blocks until the job finishes and streams its output:

```bash
flux run python3 scripts/qaoa_maxcut.py --graph-nodes 6 --max-iter 5
```

> **Note:** `--graph-nodes 6` is the *problem size* — a 6-vertex graph, i.e. 6 qubits. It has nothing to do with Flux compute nodes (this tutorial instance has **4**). Each `flux run` / `flux submit` here uses a single node and task unless you request more.

You will see each evaluation print the current cut value as COBYLA searches, followed by a summary with the best cut and the approximation ratio. The backend defaults to `local`, so nothing leaves the VM.

The script is a single file that generates a random 3-regular graph, builds the QAOA circuit, runs it on the local simulator, and drives the optimizer. Take a look:

```bash
less scripts/qaoa_maxcut.py
```

## 3. Submit non-blocking with `flux submit`

<div class="alert alert-block" style="background-color:skyblue; color:#0b2545">
<span style="font-weight:600">Description:</span> Swap <code>flux run</code> for <code>flux submit</code> to queue the job without blocking. This is closer to how you would drive many circuit evaluations in a real workflow.
</div>

```bash
# Submit without blocking; Flux returns a job ID immediately
flux submit --job-name qaoa python3 scripts/qaoa_maxcut.py --graph-nodes 6 --max-iter 5

# Watch the job in the queue
flux jobs -a

# Attach to the most recent job to see its output
flux job attach $(flux job last)
```

## 4. Scale out: an ensemble of quantum simulations with Flux

<div class="alert alert-block" style="background-color:skyblue; color:#0b2545">
<span style="font-weight:600">Description:</span> A key strength of a workload manager is running <b>many</b> independent jobs at once. Let's have Flux launch several QAOA instances — each on a different random graph — as a small ensemble.
</div>

This mirrors the bulk-submit pattern from Chapter 2, but now each Flux job drives a quantum simulation:

```bash
for seed in 1 2 3 4; do
  flux submit -N1 --job-name qaoa-$seed \
    python3 scripts/qaoa_maxcut.py --graph-nodes 6 --max-iter 3 --seed $seed
done

# See all four running/completed
flux jobs -a
```

When they finish, inspect the result of any one of them:

```bash
flux job attach $(flux jobs -a --no-header | awk 'NR==1{print $1}')
```

Here `-N1` asks Flux for one node per job, so on this 4-node instance the four jobs run one-per-node at the same time. Each job independently builds and simulates its circuits, and Flux tracks them as a group. Conceptually this is how you would fan out a parameter sweep or a batch of variational circuits from an HPC scheduler — and because it is all local, you can run as many as you like for free.

## 5. (Advanced) Run as a container in Usernetes

<div class="alert alert-block" style="background-color:skyblue; color:#0b2545">
<span style="font-weight:600">Description:</span> Optional. Run the same QAOA job as a container inside the user-space Kubernetes cluster from Chapter 4. Because everything is local, there are <b>no secrets and no credentials</b> to manage — the pod just runs the simulator.
</div>

Make sure Usernetes is running (from Chapter 4) and your kubeconfig is set:

```bash
export KUBECONFIG=/home/ubuntu/usernetes/kubeconfig
kubectl get nodes
```

We deliver the script to the pod with a ConfigMap (no image build required), then run it on a stock Python image:

```bash
# Package the script as a ConfigMap
kubectl create configmap qaoa-script \
  --from-file=qaoa_maxcut.py=scripts/qaoa_maxcut.py \
  --dry-run=client -o yaml | kubectl apply -f -

# Launch the Job (installs the SDK, then runs the local simulator)
kubectl apply -f braket-qaoa-local.yaml
```

Watch it run and read the output:

```bash
kubectl get pods -w
kubectl logs -f job/qaoa-local
```

Clean up when you are done:

```bash
kubectl delete -f braket-qaoa-local.yaml
kubectl delete configmap qaoa-script
```

The pod makes no external calls beyond pulling the base image and pip packages, and it never handles AWS credentials.

## Wrap-up

You just ran a hybrid quantum-classical optimization from an HPC workload manager, with no cloud account and no credentials:

- The **classical driver** (circuit construction + COBYLA) ran as an ordinary Flux job.
- Each **circuit evaluation** ran on the **local** quantum simulator on the VM.
- Flux scheduled a **single run**, a **non-blocking submission**, and an **ensemble** of quantum jobs.
- The same work ran as a **container in Usernetes** — still credential-free.

Thanks for following along!

Please take our 2026 survey: https://converged-computing.org/community-survey

---

### Optional appendix: running on AWS Braket SV1

If you have your own AWS Braket access and *want* to run the identical workload on the cloud SV1 state-vector simulator, the script supports it with a flag:

```bash
flux run python3 scripts/qaoa_maxcut.py --backend sv1 --graph-nodes 6 --shots 100 --max-iter 5
```

This path requires AWS credentials with Braket permissions and incurs a small per-minute cost (SV1 is billed at \$0.075/minute in `us-east-1`, with a 1-hour/month free tier for the first 12 months; a run like this is a few cents at most). It is **not** required for the tutorial — the local simulator above covers everything we teach here. An example least-privilege IAM policy scoped to only the SV1 device is included in the repository at `ec2/jupyterhub-braket-policy.json` for administrators who choose to enable it.